In [1]:
import numpy as np
from micrograd.engine import Tensor
from model import MLP
import micrograd.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer

In [18]:
losses = []

np.random.seed(37) # for reproducibility

# dataset
df = pd.read_csv('penguins.csv')



In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Species         150 non-null    str    
 1   CulmenLength    148 non-null    float64
 2   CulmenDepth     148 non-null    float64
 3   FlipperLength   148 non-null    float64
 4   OriginLocation  150 non-null    str    
 5   BodyMass        148 non-null    float64
dtypes: float64(4), str(2)
memory usage: 7.2 KB


In [20]:
df.head()

,Species,CulmenLength,CulmenDepth,FlipperLength,OriginLocation,BodyMass
0,Adelie,39.1,18.7,181.0,Torgersen,3750.0
1,Adelie,39.5,17.4,186.0,Torgersen,3800.0
2,Adelie,40.3,18.0,195.0,Torgersen,3250.0
3,Adelie,NaN,NaN,NaN,Torgersen,NaN
4,Adelie,36.7,19.3,193.0,Torgersen,3450.0


In [21]:
df['Species'] = df['Species'].map({'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2})

In [25]:
df['OriginLocation'] = df['OriginLocation'].map({'Torgersen': 0, 'Biscoe': 1, 'Dream': 2})

In [26]:
df.dropna(inplace=True)

In [28]:
df.head()

,Species,CulmenLength,CulmenDepth,FlipperLength,OriginLocation,BodyMass
0,0,39.1,18.7,181.0,0,3750.0
1,0,39.5,17.4,186.0,0,3800.0
2,0,40.3,18.0,195.0,0,3250.0
4,0,36.7,19.3,193.0,0,3450.0
5,0,39.3,20.6,190.0,0,3650.0


In [29]:
y = df['Species'].values
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [30]:
x = df.drop('Species', axis=1).values

In [31]:
x

array([[3.910e+01, 1.870e+01, 1.810e+02, 0.000e+00, 3.750e+03],
       [3.950e+01, 1.740e+01, 1.860e+02, 0.000e+00, 3.800e+03],
       [4.030e+01, 1.800e+01, 1.950e+02, 0.000e+00, 3.250e+03],
       [3.670e+01, 1.930e+01, 1.930e+02, 0.000e+00, 3.450e+03],
       [3.930e+01, 2.060e+01, 1.900e+02, 0.000e+00, 3.650e+03],
       [3.890e+01, 1.780e+01, 1.810e+02, 0.000e+00, 3.625e+03],
       [3.920e+01, 1.960e+01, 1.950e+02, 0.000e+00, 4.675e+03],
       [3.410e+01, 1.810e+01, 1.930e+02, 0.000e+00, 3.475e+03],
       [4.200e+01, 2.020e+01, 1.900e+02, 0.000e+00, 4.250e+03],
       [3.780e+01, 1.710e+01, 1.860e+02, 0.000e+00, 3.300e+03],
       [3.780e+01, 1.730e+01, 1.800e+02, 0.000e+00, 3.700e+03],
       [4.110e+01, 1.760e+01, 1.820e+02, 0.000e+00, 3.200e+03],
       [3.860e+01, 2.120e+01, 1.910e+02, 0.000e+00, 3.800e+03],
       [3.460e+01, 2.110e+01, 1.980e+02, 0.000e+00, 4.400e+03],
       [3.660e+01, 1.780e+01, 1.850e+02, 0.000e+00, 3.700e+03],
       [3.870e+01, 1.900e+01, 1.950e+02,

In [32]:
#normalize the data
scaler = StandardScaler()
x = scaler.fit_transform(x)